In [6]:
import tensorflow as tf
from tensorflow.data import TFRecordDataset 
import numpy as np
import os
import matplotlib.pyplot as plt
from common import fast_gpu_map,create_static_mask,HANNING,SAMPLERATE
from fretboard import FretBoard
from model import build_1d_cnn_model
import common
fretboard=FretBoard(17.5,SAMPLERATE)

BATCH_SIZE=1
recordfile="/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices/training_subset/filtered_data-bck.tfrecord"
model_weights="/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch82_valAcc0.9965_valPrec0.8638_valRecall0.3829.weights.h5"

cnn_model=build_1d_cnn_model(BATCH_SIZE,common.INPUT_SHAPE, common.OUTPUT_DIM_NOTES,training=False)
cnn_model.load_weights(model_weights)
dataset=TFRecordDataset(recordfile)
dataset=dataset.shuffle(1000,reshuffle_each_iteration=True)
dataset=dataset.map(lambda x: fast_gpu_map(x,training=False,threshold=0.001)).batch(BATCH_SIZE,drop_remainder=True).take(1000).prefetch(tf.data.AUTOTUNE)
#dataset=dataset.map(lambda path: fast_gpu_map(path, training=False)).take(100).prefetch(tf.data.AUTOTUNE)
#dataset = dataset.shuffle(BATCH_SIZE * 2).map(lambda path: fast_gpu_map(path, training=False), num_parallel_calls=tf.data.AUTOTUNE)
labels_true_all=[]
labels_pred_all=[]
total_tp=0
total_fp=0
total_fn=0
for audio,label in dataset:
    # print("Audio shape:", audio.shape)
    pred=cnn_model.predict(audio,verbose=0)
    # print("Pred shape:", pred.shape)
    # print("Label shape:", label.shape)
    label=label.numpy()#.flatten()
    pred=pred#.flatten()
    # print("flat Pred shape:", pred.shape)
    # print("flat Label shape:", label.shape)
    labels_true_all.append(label)
    labels_pred_all.append(pred)
    thresh=0.5
    label_active = label[:88] # Assuming 88 notes
    pred_active = pred[ :88]
    total_tp += np.sum((label_active == 1) & (pred_active > thresh))
    #total_tp+=np.sum((label==1) & (pred>thresh))
    total_fp+=np.sum((label_active==0)&(pred_active>thresh))
    total_fn+=np.sum((label_active==1)&(pred_active<=thresh))

print("Total True Positives:", total_tp)
print("Total False Positives:", total_fp)
print("Total False Negatives:", total_fn)
precision=total_tp/(total_tp+total_fp+1e-8)
recall=total_tp/(total_tp+total_fn+1e-8)
print("Precision:", precision)
print("Recall:", recall)


Initial input shape: (1, 312, 1)
After first Conv2D: (1, 156, 64)
Extracting string 1 from filters 0 to 26
String 1 section shape: (1, 26, 64)
String 1 after first Conv1D: (1, 26, 128)
Extracting string 2 from filters 26 to 52
String 2 section shape: (1, 26, 64)
String 2 after first Conv1D: (1, 26, 128)
Extracting string 3 from filters 52 to 78
String 3 section shape: (1, 26, 64)
String 3 after first Conv1D: (1, 26, 128)
Extracting string 4 from filters 78 to 104
String 4 section shape: (1, 26, 64)
String 4 after first Conv1D: (1, 26, 128)
Extracting string 5 from filters 104 to 130
String 5 section shape: (1, 26, 64)
String 5 after first Conv1D: (1, 26, 128)
Extracting string 6 from filters 130 to 156
String 6 section shape: (1, 26, 64)
String 6 after first Conv1D: (1, 26, 128)
fast gpu. Decoding
Mult hanning
Reduce max
TF.cond
fft
filtering
abs+ifft
i tensor cast
i tensor expand dims
o tensor reshape+cast
Total True Positives: 115
Total False Positives: 192
Total False Negatives: 132

I0000 00:00:1770940725.559754 2488826 local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [8]:
print("Length of labels_true_all:", len(labels_true_all))
labels_true=np.concatenate(labels_true_all,axis=0)
labels_pred=np.concatenate(labels_pred_all,axis=0)
labels_pred=(labels_pred>0.5).astype(np.float32)
print("Shape of labels true:",labels_true.shape)
#compute recall
tp=np.sum((labels_true==1) & (labels_pred==1),axis=0)

fp=np.sum((labels_true==0)&(labels_pred==1),axis=0)

print("Shape of tp",tp.shape)

precision=tp/(tp+fp+1e-8)
print("precision:", precision)
print("True Positives:", tp)
print("False Positives:", fp)

Length of labels_true_all: 1000
Shape of labels true: (1000, 129)
Shape of tp (129,)
precision: [0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.08333333
 1.         0.25       0.64285714 0.         0.         1.
 0.         0.         0.         0.         0.         0.
 0.        